# Price Optimization Model
## Modular Implementation with Parallelization

This notebook uses refactored modules for data processing, constraints, and optimization with parallel computation support.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import sys
from pathlib import Path

# Import custom modules
from data_processor import DataProcessor
from constraints import ConstraintManager
from optimization import PriceOptimizer

## 1. Configuration
Set up model parameters and file paths.

In [ ]:
# USER INPUTS
target_delta = 0.06      # Portfolio PINC
VAT = 0.19               # VAT for NR/Unit Calculation
VILC_GR = 0.0378         # VILC Annual Growth Rate

start_period = '2025-08'  # Optimization Starting Period
end_period = '2025-10'   # Optimizing Ending Period

# File paths (update these to your actual paths)
DATA_DIR = r'C:/Users/40107922/Downloads/r2'
elasticity_path = f'{DATA_DIR}/elasticity.csv'
reference_path = f'{DATA_DIR}/reference_abi_sellin-vol_pl-ptc.csv'
competitor_elasticity_path = f'{DATA_DIR}/elasticity_competitor.csv'
competitor_reference_path = f'{DATA_DIR}/reference_comp_sellout-vol_ptc.csv'
seg_mapping_path = f'{DATA_DIR}/segment_mapping.csv'

# Optimization settings
n_jobs = -1  # Use all available cores for parallelization
tolerance = 0.005  # Tolerance for PINC constraint

## 2. Data Processing
Load and prepare all data using the DataProcessor module.

In [ ]:
# Initialize data processor
data_processor = DataProcessor(
    min_period='2024-01',
    max_period='2026-12',
    vilc_gr=VILC_GR
)

# Load and process all data
print("Loading and processing data...")
(reference_df, competitor_reference_df, E_price_to_volume, 
 E_price_to_comp_volume, metadata) = data_processor.load_and_process_data(
    elasticity_path=elasticity_path,
    reference_path=reference_path,
    competitor_elasticity_path=competitor_elasticity_path,
    competitor_reference_path=competitor_reference_path,
    seg_mapping_path=seg_mapping_path,
    start_period=start_period,
    end_period=end_period
)

print(f"Data processing complete!")
print(f"Number of own products: {metadata['num_own']}")
print(f"Number of competitor products: {metadata['num_comp']}")
print(f"Number of months: {metadata['num_months']}")
print(f"Months: {metadata['months']}")

## 3. Initialize Optimizer
Create the PriceOptimizer instance with parallelization support.

In [ ]:
# Initialize optimizer
optimizer = PriceOptimizer(
    reference_df=reference_df,
    competitor_reference_df=competitor_reference_df,
    own_products=metadata['own_products'],
    competitor_products=metadata['competitor_products'],
    months=metadata['months'],
    num_months=metadata['num_months'],
    num_own=metadata['num_own'],
    num_comp=metadata['num_comp'],
    E_price_to_volume=E_price_to_volume,
    E_price_to_comp_volume=E_price_to_comp_volume,
    VAT=VAT,
    # n_jobs=n_jobs
)

print(f"Optimizer initialized with {n_jobs} parallel jobs")

## 4. Set Up Constraints
Create optimization constraints using the ConstraintManager.

In [ ]:
# Initialize constraint manager
constraint_manager = ConstraintManager(
    reference_df=reference_df,
    competitor_reference_df=competitor_reference_df,
    own_products=metadata['own_products'],
    competitor_products=metadata['competitor_products'],
    months=metadata['months'],
    num_months=metadata['num_months'],
    num_own=metadata['num_own'],
    num_comp=metadata['num_comp'],
    E_price_to_volume=E_price_to_volume,
    E_price_to_comp_volume=E_price_to_comp_volume,
    target_delta=target_delta,
    VAT=VAT,
    get_reference_arrays_own_func=optimizer.get_reference_arrays_own,
    get_reference_arrays_comp_func=optimizer.get_reference_arrays_comp,
    calc_volume_func=optimizer.calc_volume,
    calc_MACO_func=optimizer.calc_MACO
)

# Create constraints
constraints = constraint_manager.create_constraints(tolerance=tolerance)
print(f"Created {len(constraints)} constraints")

## 5. Set Up Bounds and Initial Guess
Define price bounds and initial price guess.

In [ ]:
# Create bounds
bounds = optimizer.create_bounds(metadata['reference_df_padded_bound'])
print(f"Created bounds for {len(bounds)} decision variables")

# Setup initial guess for prices
reference_df_ordered = reference_df.set_index(['sku', 'year_month']).loc[
    [(sku, month) for month in metadata['months'] 
     for sku in metadata['own_products']]
].reset_index()

reference_df_ordered['reference_price_unit'] = (
    reference_df_ordered['reference_price'] * 
    reference_df_ordered['capacity'] / 1000
)

P0 = reference_df_ordered['reference_price_unit'].values
print(f"Initial guess shape: {P0.shape}")

## 6. Run Optimization
Execute the optimization with parallelization.

In [ ]:
# Run optimization
(result, monthly_outputs_unrounded, industry_volume_unrounded,
 rounded_prices, monthly_outputs_rounded, industry_volume_rounded) = optimizer.optimize(
    constraints=constraints,
    bounds=bounds,
    P0=P0,
    method='trust-constr',
    options={'disp': True}
)

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE")
print("="*60)
print(f"Success: {result.success}")
print(f"Message: {result.message}")
print(f"Objective value: {result.fun:.2f}")
print(f"Number of iterations: {result.nit}")

## 7. Results Validation
Check constraint adherence and calculate key metrics.

In [ ]:
def validate_results(monthly_df, industry_df, target_delta, tolerance=0.005):
    """Validate optimization results against constraints."""
    
    # Calculate total volumes
    total_ref_volume = industry_df['volume_ref'].sum()
    total_opt_volume = industry_df['volume_opt'].sum()
    abi_ref_volume = industry_df[industry_df['manufacturer']=='abi']['volume_ref'].sum()
    abi_opt_volume = industry_df[industry_df['manufacturer']=='abi']['volume_opt'].sum()
    
    print("\n" + "="*60)
    print("RESULTS VALIDATION")
    print("="*60)
    
    # MACO
    print("\n📊 TOTAL MACO:")
    total_maco_reference = monthly_df['MACO_ref'].sum()
    total_maco_optimized = monthly_df['MACO_opt'].sum()
    print(f"  Reference MACO: {total_maco_reference:,.0f}")
    print(f"  Optimized MACO: {total_maco_optimized:,.0f}")
    print(f"  MACO change: {total_maco_optimized/total_maco_reference - 1:,.4f}")
    if total_maco_optimized < total_maco_reference:
        print("  ❌ MACO declined after optimization.")
    else:
        print("  ✅ MACO grew after optimization.")
    
    # Industry volume
    print("\n🏭 INDUSTRY VOLUME CONSTRAINT:")
    print(f"  Total industry reference volume: {total_ref_volume:,.2f}")
    print(f"  Total industry optimized volume: {total_opt_volume:,.2f}")
    print(f"  Total industry volume change: {total_opt_volume/total_ref_volume - 1:,.4f}")
    
    lower_bound = 0.99 * (total_ref_volume * (1 - 0.56*target_delta))
    if total_opt_volume < lower_bound:
        print("  ❌ Constraint violated: industry volume decreased too much.")
    else:
        print("  ✅ Constraint satisfied: industry volume within bounds.")
    
    # Own volume
    print("\n🏢 OWN VOLUME CONSTRAINT:")
    print(f"  Total ABI reference volume: {abi_ref_volume:,.2f}")
    print(f"  Total ABI optimized volume: {abi_opt_volume:,.2f}")
    print(f"  Total ABI volume change: {abi_opt_volume/abi_ref_volume - 1:,.4f}")
    
    lower_bound = 0.99 * abi_ref_volume
    upper_bound = 1.05 * abi_ref_volume
    if abi_opt_volume < lower_bound:
        print("  ❌ Constraint violated: ABI volume decreased too much.")
    elif abi_opt_volume > upper_bound:
        print("  ❌ Constraint violated: ABI volume increased too much.")
    else:
        print("  ✅ Constraint satisfied: ABI volume within bounds.")
    
    # Market share
    print("\n📈 MARKET SHARE CONSTRAINT:")
    ref_ms = abi_ref_volume / total_ref_volume
    opt_ms = abi_opt_volume / total_opt_volume
    print(f"  Reference Market Share: {ref_ms:,.4f}")
    print(f"  Optimized Market Share: {opt_ms:,.4f}")
    print(f"  Market Share Change: {opt_ms - ref_ms:,.4f}")
    
    if opt_ms < ref_ms - 0.005:
        print("  ❌ Constraint violated: market share decreased too much.")
    else:
        print("  ✅ Constraint satisfied: market share within bounds.")
    
    # Portfolio PINC
    print("\n💰 PORTFOLIO PINC CONSTRAINT:")
    ref_ppl = np.sum(monthly_df['volume_ref'] * monthly_df['price_liter_ref']) / np.sum(monthly_df['volume_ref'])
    opt_ppl = np.sum(monthly_df['volume_opt'] * monthly_df['price_liter_opt']) / np.sum(monthly_df['volume_opt'])
    pinc_delta = (opt_ppl / ref_ppl) - 1
    
    print(f"  Reference Portfolio Price/Liter: {ref_ppl:,.4f}")
    print(f"  Optimized Portfolio Price/Liter: {opt_ppl:,.4f}")
    print(f"  Specified PINC: {target_delta:,.4f}")
    print(f"  Optimized PINC: {pinc_delta:,.4f}")
    
    if np.abs(pinc_delta - target_delta) > tolerance:
        print(f"  ❌ Constraint violated: PINC deviation too large.")
    else:
        print(f"  ✅ Constraint satisfied: PINC within tolerance.")
    
    # Hierarchy
    print("\n🏆 NR/HL HIERARCHY CONSTRAINT:")
    segment_order = ['Value', 'Core', 'Core+', 'Premium', 'Super Premium']
    segment_grouped = monthly_df.groupby('segment').agg({
        'NR_opt': 'sum',
        'volume_opt': 'sum'
    })
    segment_grouped['NR_per_HL'] = segment_grouped['NR_opt'] / segment_grouped['volume_opt']
    segment_nrh_agg = segment_grouped['NR_per_HL'].reindex(segment_order)
    
    size_order = ['Small', 'Regular', 'Large']
    size_grouped = monthly_df.groupby('size_group').agg({
        'NR_opt': 'sum',
        'volume_opt': 'sum'
    })
    size_grouped['NR_per_HL'] = size_grouped['NR_opt'] / size_grouped['volume_opt']
    size_nrh_agg = size_grouped['NR_per_HL'].reindex(size_order)
    
    # Check segment hierarchy
    violated_segments = []
    for i in range(len(segment_order) - 1):
        if segment_nrh_agg[segment_order[i]] > segment_nrh_agg[segment_order[i + 1]]:
            violated_segments.append((segment_order[i], segment_order[i + 1]))
    
    if violated_segments:
        print("  ❌ Segment NR/HL hierarchy violated between:")
        for seg1, seg2 in violated_segments:
            print(f"     {seg1} > {seg2}")
    else:
        print("  ✅ Segment NR/HL hierarchy is satisfied")
    
    # Check size hierarchy
    violated_sizes = []
    for i in range(len(size_order) - 1):
        if size_nrh_agg[size_order[i]] < size_nrh_agg[size_order[i + 1]]:
            violated_sizes.append((size_order[i], size_order[i + 1]))
    
    if violated_sizes:
        print("  ❌ Size group NR/HL hierarchy violated between:")
        for sg1, sg2 in violated_sizes:
            print(f"     {sg1} < {sg2}")
    else:
        print("  ✅ Size group NR/HL hierarchy is satisfied")
    
    print("\n" + "="*60)

# Validate unrounded results
print("\n### UNROUNDED RESULTS ###")
validate_results(monthly_outputs_unrounded, industry_volume_unrounded, 
                target_delta, tolerance)

# Validate rounded results
print("\n### ROUNDED RESULTS (Multiples of 50) ###")
validate_results(monthly_outputs_rounded, industry_volume_rounded, 
                target_delta, tolerance)

## 8. Export Results
Save optimized prices and volumes to files.

In [ ]:
# Export results
output_dir = Path('optimization_results')
output_dir.mkdir(exist_ok=True)

# Export monthly outputs
monthly_outputs_rounded.to_excel(
    output_dir / 'monthly_optimization_output.xlsx', 
    index=False
)
print(f"✅ Saved monthly outputs to {output_dir / 'monthly_optimization_output.xlsx'}")

# Export industry volumes
industry_volume_rounded.to_excel(
    output_dir / 'industry_volume_output.xlsx', 
    index=False
)
print(f"✅ Saved industry volumes to {output_dir / 'industry_volume_output.xlsx'}")

# Export optimization summary
summary_df = pd.DataFrame({
    'Metric': [
        'Optimization Success',
        'Objective Value',
        'Iterations',
        'Target PINC',
        'Achieved PINC',
        'MACO Improvement',
        'ABI Volume Change',
        'Industry Volume Change'
    ],
    'Value': [
        result.success,
        f"{result.fun:.2f}",
        result.nit,
        f"{target_delta:.4f}",
        f"{(np.sum(monthly_outputs_rounded['volume_opt'] * monthly_outputs_rounded['price_liter_opt']) / np.sum(monthly_outputs_rounded['volume_opt']) / (np.sum(monthly_outputs_rounded['volume_ref'] * monthly_outputs_rounded['price_liter_ref']) / np.sum(monthly_outputs_rounded['volume_ref'])) - 1):.4f}",
        f"{(monthly_outputs_rounded['MACO_opt'].sum() / monthly_outputs_rounded['MACO_ref'].sum() - 1):.4f}",
        f"{(industry_volume_rounded[industry_volume_rounded['manufacturer']=='abi']['volume_opt'].sum() / industry_volume_rounded[industry_volume_rounded['manufacturer']=='abi']['volume_ref'].sum() - 1):.4f}",
        f"{(industry_volume_rounded['volume_opt'].sum() / industry_volume_rounded['volume_ref'].sum() - 1):.4f}"
    ]
})

summary_df.to_excel(output_dir / 'optimization_summary.xlsx', index=False)
print(f"✅ Saved optimization summary to {output_dir / 'optimization_summary.xlsx'}")

print("\n✅ All results exported successfully!")